<a href="https://colab.research.google.com/github/Shaxzod1991/ABC_Analysis/blob/main/PL_analyze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import glob
import os
!pip install duckdb
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ***Пути к входным и выходным данным***



In [47]:
path_cost_of_sale = r'/content/sample_data/91_счет'                             #Путь к файлам себестоимости
path_revenue = r'/content/sample_data/90_счет'                                  #Путь к файлам Выручки
path_overheads = r'/content/sample_data/94_счет'                                #Путь к файлам расходов периода
path_other_income = r'/content/sample_data/90_счет/93_счет'
path_output = r'/content/drive/MyDrive/Проекты_Санег/Коды_только/Результаты'

# ***Функции преобразования и обработки данных***

In [19]:
#______________________________________________Функция для изменения на тип "Дата"______________________________________________
def def_time_type(df, col):
    df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', dayfirst=True, errors = 'coerce')
    return df

#______________________________________________Функция для изменения типов чисел________________________________________________
def def_num_type(df, col):
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace(r'\s+', '', regex = True)
    df[col] = df[col].str.replace(',', '.', regex = True)
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

#______________________________Функция для объединения файлов выгрузок из 1С в формате txt._____________________________________
def def_load_and_combine(path_name, search_key = 'Период'):
    file_paths = glob.glob(os.path.join(path_name, '*txt'))
    all_frame = []

    for f_path in file_paths:

      line_skip_row = 0
      with open(f_path, 'r', encoding = 'utf - 8') as f_obj:
        for numb, row_name in enumerate(f_obj):
          if search_key in row_name:
            line_skip_row = numb
            break

      df = pd.read_csv(f_path, encoding = 'utf-8', skiprows=line_skip_row, sep = '\t')
      all_frame.append(df)

    if all_frame:                                                        # Объединяем всё, если файлы были найдены
          combined_df = pd.concat(all_frame, ignore_index=True)
          return combined_df
    else:
          print(f"Предупреждение: В папке '{path_name}' файлы .txt не найдены.")
          return pd.DataFrame()                                          # Возвращаем пустой DataFrame, чтобы код не падал дальше

#__________________________________________Функция для переименовании названий столбцов ___________________________________________
def def_col_rename(df):

  columns_name = {'Период': 'Дата',
                'Документ': 'Документ',
                'Аналитика Дт': 'Аналитика_Дт',
                'Аналитика Кт': 'Аналитика_Кт',
                'Дебет': 'Счет_Дт',
                'Кредит': 'Счет_Кт',
                'Unnamed: 6': 'Сумма_Дт',
                'Unnamed: 9': 'Сумма_Кт'}

  df = df.rename(columns=columns_name)
  return df

# ***Себестоимость (Cost of Sales). Объединение и преобразование выгрузок***

In [20]:
#______________________________Применение функции к файлам себестоимости _____________________________________________________
cdf_cost_of_sale = def_load_and_combine(path_cost_of_sale, search_key = 'Период')
cdf_cost_of_sale = def_col_rename(cdf_cost_of_sale)
cdf_cost_of_sale = def_time_type(cdf_cost_of_sale, 'Дата')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Дт')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Кт')

## ***Обработка себестоимости c использованием DuckDB SQL***

In [21]:
#__________________________________________________Обработка в SQL ____________________________________________________________

# 1. Удаляем старую таблицу, если она осталась, и создаем заново
duckdb.execute("DROP TABLE IF EXISTS cost_of_sale")

create_empty_table = """
CREATE TEMPORARY TABLE cost_of_sale (
    "Дата" DATE,
    "Номер_документа" VARCHAR,
    "Аналитика" VARCHAR,
    "Счет_Дт" VARCHAR,
    "Счет_Кт" VARCHAR,
    "Себес_Сумма" DECIMAL(20,2),
    "Кол-во" DECIMAL(10,5)
);
"""
duckdb.execute(create_empty_table)

insert_query = """
INSERT INTO cost_of_sale
   WITH t1 AS (
    SELECT
        *,
        CASE
            WHEN "Показатель" = 'БУ' AND LEFT("Счет_Дт", 2) = '91' THEN lead("Сумма_Кт") OVER()
            ELSE (lead("Сумма_Кт") OVER())*-1
        END AS "Кол-во"
    FROM cdf_cost_of_sale
    WHERE "Показатель" IN ('БУ', 'Кол.')
)
SELECT
    "Дата",
    STRING_SPLIT("Документ", CHR(10))[1] AS "Номер_документа",
    CASE
       WHEN LEFT("Счет_Дт", 2) = '91' AND "Аналитика_Дт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[2]
        WHEN LEFT("Счет_Кт", 2) = '91' AND "Аналитика_Кт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[2]
    END AS "Аналитика",
    "Счет_Дт",
    "Счет_Кт",
    CASE
        WHEN LEFT("Счет_Дт", 2) = '91' THEN "Сумма_Дт"
        WHEN LEFT("Счет_Кт", 2) = '91' THEN "Сумма_Кт" * -1
        ELSE 0
    END AS "Себес_Сумма",
    "Кол-во"
FROM t1
WHERE "Дата" IS NOT NULL AND "Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%'
ORDER BY "Дата"
"""
duckdb.execute(insert_query)

# 2. Выполняем запрос через DuckDB и сразу превращаем результат в новый датафрейм
cost_of_sale = duckdb.query("SELECT * FROM cost_of_sale").df()

pd.concat([cost_of_sale.head(5), cost_of_sale.tail(5)])

,Дата,Номер_документа,Аналитика,Счет_Дт,Счет_Кт,Себес_Сумма,Кол-во
0,2025-01-01,Реализация товаров и услуг 00000000052 от 01.0...,"Дизельное топливо ТД 0,5-40",9110.51,2810.2,"172,798,638.01",57.90
1,2025-01-02,Реализация товаров и услуг 00000000020 от 02.0...,"Дизельное топливо ТД 0,5-40",9110.52,2810.2,"160,721,154.56",57.14
2,2025-01-02,Реализация товаров и услуг 00000000021 от 02.0...,"Дизельное топливо ТД 0,5-40",9110.52,2810.2,"161,339,961.95",57.36
3,2025-01-03,Реализация товаров и услуг 00000000002 от 03.0...,"Дизельное топливо ТД 0,5-40",9110.52,2810.2,"25,042,060.23",8.37
4,2025-01-03,Реализация товаров и услуг 00000000003 от 03.0...,"Дизельное топливо ТД 0,5-40",9110.52,2810.2,"25,006,170.50",8.36
9389,2026-03-31,Операция (бухгалтерский учет) 00000001741 от 3...,Возмещение затрат (другие),9130.90,2310.5,"13,294,981.48",NaN
9390,2026-03-31,Операция (бухгалтерский учет) 00000001741 от 3...,Возмещение затрат (другие),9130.90,2310.5,"112,151,534.38",NaN
9391,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"3,064,202.72",NaN
9392,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"3,716,277.32",NaN
9393,2026-03-31,Операция (бухгалтерский учет) 00000001742 от 3...,Возмещение затрат (другие),9130.90,2310.5,"30,968,977.68",NaN


# ***Выручка (Revenue). Объединение и преобразование выгрузок***

In [22]:
#_________________________________________Применение функции к файлам себестоимости ___________________________________________
cdf_revenue = def_load_and_combine(path_revenue, search_key = 'Период')
cdf_revenue = def_col_rename(cdf_revenue)
cdf_revenue = def_time_type(cdf_revenue, 'Дата')
cdf_revenue = cdf_revenue.rename(columns={'Unnamed: 5': 'Сумма_Дт', 'Unnamed: 8': 'Сумма_Кт', 'Сумма_Дт': 'Удалить_1', 'Сумма_Кт': 'Удалить_2'})
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Дт')
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Кт')

## ***Обработка выручки с использованием DuckDB SQL***

In [23]:
#__________________________________________________Обработка в SQL ____________________________________________________________

#Удаляем старую таблицу, если она осталась, и создаем заново
duckdb.execute("DROP TABLE IF EXISTS revenue_SQL")

create_empty_table_revenue = """
    CREATE TEMPORARY TABLE revenue_SQL (
        "Дата" DATE,
        "Номер_документа" VARCHAR,
        "Аналитика" VARCHAR,
        "Счет_Дт" VARCHAR,
        "Счет_Кт" VARCHAR,
        "Выручка_Сумма" DECIMAL(20,2)
        )
"""
duckdb.execute(create_empty_table_revenue)

duckdb.register("duck_cdf_revenue", cdf_revenue)

insert_query_revenue = """
    INSERT INTO revenue_SQL
    SELECT
      "Дата",
       CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
      END AS "Номер_документа",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[1]
      END AS "Аналитика",
      "Счет_Дт",
      "Счет_Кт",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN "Сумма_Дт" * -1
        WHEN "Счет_Кт" LIKE '90%' THEN "Сумма_Кт"
      END AS "Выручка_Сумма"
    FROM duck_cdf_revenue
    WHERE ("Счет_Дт" LIKE '90%' OR "Счет_Кт" LIKE '90%') AND ("Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%')
"""
duckdb.execute(insert_query_revenue)

revenue = duckdb.query("select * from revenue_SQL").df()


# ***Объединение данных по выручке и себестоимости с использованием DuckDB SQL***

In [24]:
cost_of_sale['Аналитика'] = cost_of_sale['Аналитика'].str.strip()
revenue['Аналитика'] = revenue['Аналитика'].str.strip()

#__________________________________________________Обработка в SQL ____________________________________________________________
duckdb.execute("DROP TABLE IF EXISTS group_rev")

group_rev = """
    CREATE TEMPORARY TABLE group_rev AS
    SELECT
      "Дата",
      "Аналитика",
      SUM("Выручка_Сумма") AS "Выручка_Сумма"
    FROM revenue
    GROUP BY
      "Дата",
      "Аналитика"
"""
duckdb.execute(group_rev)

pivot_query = """
    with t1 as (select
      "Дата",
      "Аналитика",
      SUM("Себес_Сумма") AS "Себес_Сумма",
      SUM("Кол-во") AS "Кол-во"
    from cost_of_sale
    group by
      "Дата",
      "Аналитика"),
       t2 as (select
        *
      from t1
      full join group_rev on t1."Дата" = group_rev."Дата" and t1."Аналитика" = group_rev."Аналитика")
        select
          CASE
            when "Дата" is null then "Дата_1"
            else "Дата"
          END AS "Дата",
            CASE
                when "Аналитика" is null then "Аналитика_1"
                else "Аналитика"
            END as "Аналитика",
          "Выручка_Сумма",
          "Себес_Сумма",
          "Кол-во"
        from t2
        order by "Дата", "Выручка_Сумма"
        """
reven_cost_of_sale = duckdb.query(pivot_query).df()


## ***Справочик статьи выручки и себестоимости***

In [42]:
dict_directory_revenue_cols = {
    "Авиакеросин ТС-1": "Авиакеросин",
    "Авиационное топливо JET A-1": "Авиакеросин",
    "Авиационный бензин Б-92": "Бензин автомобильный АИ-92-К-2-Л",
    "Автобензин АИ 95 К2-Л": "Бензин АИ-95",
    "Автомобильный бензин АИ-92-К5 (импорт)": "Бензин автомобильный АИ-92-К-2-Л",
    "Автомобильный бензин АИ-95-К2-Л": "Бензин АИ-95",
    "Автомобильный бензин АИ-95-К5 (импорт)": "Бензин АИ-95",
    "Аренда услуга": "Прочие",
    "Бензин автомобильный АИ-80-К2-Л (местный)": "Бензин АИ-80 К-2Л",
    "Бензин автомобильный АИ-92-К-2-Л": "Бензин автомобильный АИ-92-К-2-Л",
    "Бензин экстракционный прямагонный": "Экстра бензин",
    "Битум нефтяной дорожный БНД 60/90": "Нефтебитум",
    "Битум нефтяной строительный БН 40/60": "Нефтебитум",
    "Битум нефтяной строительный БН 90/10": "Нефтебитум",
    "Возмещение затрат (другие)": "Прочие",
    "Возмещение затрат (транспорт АВТО)": "Прочие",
    "Возмещение затрат (транспорт ЖД)": "Прочие",
    "Газовый конденсат": "Газ и прочие газы",
    "Гач II фракции марки 1": "Прочие",
    "Дизельное топливо ЕВРО Л - К4": "Дизтопливо Евро 5",
    "Дизельное топливо З-2 (Зим)": "Дизтопливо Евро 5",
    "Дизельное топливо ТД 0,5": "Дизельное топливо ГОСТ 0,5",
    "Дизельное топливо ТД 0,5-40": "Дизельное топливо ГОСТ 0,5",
    "Дизельное топливо ЭКО Л-0,035-40": "Дизтопливо Евро 5",
    "Керосин авиационный (JET A-1)-SSF": "Авиакеросин",
    "Кокс нефтяной высокосернистый суммарный": "Нефтекокс",

    "Масло SN-100": "Нефтемасла",
    "Масло SN-150": "Нефтемасла",
    "Масло SN-500 (ФНПЗ)": "Нефтемасла",
    "Масло SN-900 (ФНПЗ)": "Нефтемасла",
    "Масло базовое SN-350": "Нефтемасла",
    "Масло дистиллятное I фракции": "Нефтемасла",
    "Масло дистиллятное II фракции": "Нефтемасла",
    "Масло дистиллятное III фракции": "Нефтемасла",
    "Масло индустриальное И12А": "Нефтемасла",
    "Масло индустриальное И20А": "Нефтемасла",
    "Масло индустриальное И40А": "Нефтемасла",
    "Масло индустриальное И50А": "Нефтемасла",
    "Масло компрессорное КС-19 (ФНПЗ)": "Нефтемасла",
    "Масло моторное М-14В2": "Нефтемасла",
    "Масло моторное М20А": "Нефтемасла",
    "Масло моторное универсальное М-8B(ФНПЗ)": "Нефтемасла",
    "Масло мотроное М14Г2К": "Нефтемасла",
    "Масло остаточное": "Нефтемасла",
    "Масло трансмиссионное Ферганол ТМ-2-34 (SAE 140 API GL-2)": "Нефтемасла",
    "Масло трансформаторное селективной очистки ТРМ-1": "Нефтемасла",
    "Масло Турбинное с присадками ТП - 30": "Нефтемасла",
    "Турбинное масло ТП-22": "Нефтемасла",
    'Фракция легких углеводородов" Ts 05767930-250:2022': "Авиакеросин",

    "Нефрас": "Прочие",
    "Нефть SANEG": "Нефть",
    "Нефтяной шлам": "Прочие",

    "Общие GRDC": "Прочие",
    "Общие ТПП Андижан": "Прочие",
    "Общие ТСП SANEG Construction": "Прочие",

    "ПЕТРОЛАТУМ Марки ПС 55": "Прочие",
    "ПЕТРОЛАТУМ Марки ПС 60": "Прочие",

    "Питание EMS Андижан": "Прочие",
    "Питание EMS Карши": "Прочие",
    "Питание EMS Мубарек": "Прочие",
    "Питание Автонефттранс Андижан": "Прочие",
    "Питание Автонефттранс Карши": "Прочие",
    "Питание Автонефттранс Мубарек": "Прочие",
    "Питание Петротек Азия": "Прочие",
    "Питание ЧП Махсус мелиорация курилиш ва таъмирлаш": "Прочие",

    "Попутный (нефтяной) газ": "Газ и прочие газы",
    "Природный газ": "Природный газ",
    "Прочие ТМЦ и активы": "Прочие",
    "Разовые услуги": "Прочие",

    "Сера техническая (ФНПЗ)": "Прочие",
    "Сера техническая газовая комовая": "Прочие",

    "Сжиженный газ (ФНПЗ)": "Сжиженный газ",
    "Сжиженный углеводородный газ (пропан-бутан)": "Сжиженный газ",
    "Сжиженый газ МГПЗ": "Сжиженный газ",
    "Сжиженый газ Шуртан НГДУ": "Сжиженный газ",

    "Скважина №1 Кокчалак": "Прочие",
    "Скважина №1002 Газли": "Прочие",
    "Скважина №1004 Газли": "Прочие",
    "Скважина №1012 Газли": "Прочие",
    "Скважина №1018 Газли": "Прочие",
    "Скважина №1021 Газли": "Прочие",
    "Скважина №1022 Газли": "Прочие",
    "Скважина №1026 Газли": "Прочие",
    "Скважина №11 Акчалак": "Прочие",
    "Скважина №1101 Газли": "Прочие",
    "Скважина №1111 Газли": "Прочие",
    "Скважина №1123 Газли": "Прочие",
    "Скважина №1125 Газли": "Прочие",
    "Скважина №1130 Газли": "Прочие",
    "Скважина №1141 Газли": "Прочие",
    "Скважина №119 Газли": "Прочие",
    "Скважина №1192 Газли": "Прочие",
    "Скважина №1196 Газли": "Прочие",
    "Скважина №122 Газли": "Прочие",
    "Скважина №1224 Газли": "Прочие",
    "Скважина №1225 Газли": "Прочие",
    "Скважина №1242 Газли": "Прочие",
    "Скважина №1252 Газли": "Прочие",
    "Скважина №1256 Газли": "Прочие",
    "Скважина №144 Газли": "Прочие",
    "Скважина №181 Газли": "Прочие",
    "Скважина №187 Газли": "Прочие",
    "Скважина №204 Газли": "Прочие",
    "Скважина №217 Газли": "Прочие",
    "Скважина №235 Газли": "Прочие",
    "Скважина №242 Газли": "Прочие",
    "Скважина №247 Газли": "Прочие",
    "Скважина №255 Газли": "Прочие",
    "Скважина №27 Газли": "Прочие",
    "Скважина №271 Газли": "Прочие",
    "Скважина №273 Газли": "Прочие",
    "Скважина №291 Газли": "Прочие",
    "Скважина №3 Кокчалак": "Прочие",
    "Скважина №316 Газли": "Прочие",
    "Скважина №326 Газли": "Прочие",
    "Скважина №329 Газли": "Прочие",
    "Скважина №352 Газли": "Прочие",
    "Скважина №368 Газли": "Прочие",
    "Скважина №390 Газли": "Прочие",
    "Скважина №455 Газли": "Прочие",
    "Скважина №46 Урга": "Прочие",
    "Скважина №5 Акчалак": "Прочие",
    "Скважина №6 Акчалак": "Прочие",
    "Скважина №601 Газли": "Прочие",
    "Скважина №809 Газли": "Прочие",
    "Скважина №818 Газли": "Прочие",
    "Скважина №827 Газли": "Прочие",
    "Скважина №92 Газли": "Прочие",
    "Скважина №98 Газли": "Прочие",

    "Топливо нефтяное мазут М 100": "Мазут",
    "Топливо нефтяное мазут М 40": "Мазут",
    "Топливо печное бытовое": "Печное топливо",

    "Транспортные услуги": "Прочие",
    "Услуги": "Прочие",
    "Услуги по организации питания": "Прочие",

    "Фракция легких углеводородов Ts 05767930-250:2022": "ФЛУ",
    "Хранение ГСМ": "Прочие",
    "Экстракт II фракции": "Экстра бензин"
}


## ***Добавление анатилики свода по выручке и себестоимомти***

In [43]:
reven_cost_of_sale['Аналитика_Свод'] = reven_cost_of_sale['Аналитика'].map(dict_directory_revenue_cols)

# ***Расходы периода. Обработка***

In [25]:
df_overheads = def_load_and_combine(path_overheads, search_key = 'Период')

def_time_type(df_overheads, 'Период')
def_num_type(df_overheads, 'Unnamed: 5')
def_num_type(df_overheads, 'Unnamed: 8')
df_overheads = df_overheads.iloc[:, [0, 1, 2, 3,4,5,7,8]]
df_overheads = df_overheads.rename(columns={'Период': 'Дата','Аналитика Дт': 'Аналитика_Дт', 'Аналитика Кт': 'Аналитика_Кт' , 'Дебет': 'Счет_Дт','Unnamed: 5': 'Сумма_Дт', 'Кредит': 'Счет_Кт' ,'Unnamed: 8': 'Сумма_Кт'})
df_overheads['Сумма_Дт'] = df_overheads['Сумма_Дт'].fillna(0)
df_overheads['Сумма_Кт'] = df_overheads['Сумма_Кт'].fillna(0)

## ***Обработка расходов периода в DuckDB SQL***

In [26]:
query_overh = """

    WITH T1 AS (SELECT
        *,
        CASE
            WHEN "Счет_Дт" LIKE '9410%' OR "Счет_Кт" LIKE '9410%' THEN '9410'
            WHEN "Счет_Дт" LIKE '9420%' OR "Счет_Кт" LIKE '9420%' THEN '9420'
            WHEN "Счет_Дт" LIKE '9430%' OR "Счет_Кт" LIKE '9430%' THEN '9430'
        END AS "Счет_Загрузки",
        CASE
            WHEN "Счет_Дт" LIKE '94%' THEN STRING_SPLIT(REPLACE("Аналитика_Дт", CHR(13), ''), CHR(10))[3]
            WHEN "Счет_Кт" LIKE '94%' THEN STRING_SPLIT(REPLACE("Аналитика_Кт", CHR(13), ''), CHR(10))[3]
        END AS "Аналитика"
        FROM df_overheads
        WHERE
            "Дата" IS NOT NULL AND ("Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%')
        )
            SELECT
                "Дата",
                "Аналитика_Дт",
                "Аналитика_Кт",
                "Счет_Дт",
                "Счет_Кт",
                "Счет_Загрузки",
                CASE
                    WHEN "Счет_Загрузки" LIKE '9410' THEN 'Коммерческие расходы'
                    WHEN "Счет_Загрузки" LIKE '9420' THEN 'Административные расходы'
                    WHEN "Счет_Загрузки" LIKE '9430' THEN 'Прочие расходы'
                END AS "Название_Статьи",
                CASE
                    WHEN "Аналитика" LIKE '<...>' AND "Счет_Кт" LIKE '92%' THEN 'Списание ОС'
                    WHEN "Аналитика" LIKE '<...>' THEN STRING_SPLIT(REPLACE("Документ", CHR(13), ''), CHR(10))[2]
                    ELSE "Аналитика"
                END AS "Аналитика",
                "Сумма_Дт" - "Сумма_Кт" AS "Сумма"
            FROM T1
"""

overheads = duckdb.sql(query_overh).df()

## ***Справочник статьи затрат для Расходов периода***

In [33]:
expense_dict = {
    "Прочие расходы": "Прочие операционные и внереализационные расходы",
    "Услуги по жд": "Транспорт и логистика",
    "Таможенные сборы по экспорту": "Таможенные и государственные услуги",
    "Расходы прошлых лет": "Расходы прошлых лет",
    "Штрафы, пени, неустойки и санкций за нарушение условий договоров": "Штрафы, санкции, страхование и судебные расходы",
    "Перемещена амортизация ОС на новый счет": "Амортизация и внеоборотные активы",
    "Прочие расходы (не относящиеся к основной деятельности)": "Прочие операционные и внереализационные расходы",
    "Возмещение за учебу (УХТА)": "Расходы на персонал",
    "Штрафные санкции по ТехПД": "Штрафы, санкции, страхование и судебные расходы",
    "Содержание железнодорожных путей": "Ремонт и содержание имущества",
    "Командировочные расходы": "Командировочные и представительские расходы",
    "Топливо": "Материалы и производственное обеспечение",
    "Ремонт и техническое обслуживание автотранпорта": "Ремонт и содержание имущества",
    "Ремонт прочего оборудования": "Ремонт и содержание имущества",
    "Проезд в командировках": "Командировочные и представительские расходы",
    "государственные услуги": "Таможенные и государственные услуги",
    "Проживание в командировках": "Командировочные и представительские расходы",
    "Услуги по содержанию территорий и помещений": "Ремонт и содержание имущества",
    "09510 Неотложка хисобидан махсус сувдан фойдаланиг ёки сувни истеъмол килиш учун": "Налоги, сборы и обязательные платежи",
    "Консалтинговые услуги": "Консультационные и экспертные услуги",
    "Вахтовая перевозка": "Транспорт и логистика",
    "Прочие финансовые расходы": "Финансовые и банковские расходы",
    "Хранение сырья, материалов, готового продукта, ОС": "Транспорт и логистика",
    "Экспертиза": "Консультационные и экспертные услуги",
    "Реклама": "Прочие операционные и внереализационные расходы",
    "Банковские услуги": "Финансовые и банковские расходы",
    "Комиссионное сбор и клиринговое обслуживание Уз РТСБ": "Финансовые и банковские расходы",
    "Аренда жилых помещений": "Аренда и содержание объектов",
    "Юридические консультационные услуги": "Консультационные и экспертные услуги",
    "Внедрение и сопровождение программного продукта": "ИТ и связь",
    "Техника и мебель": "Материалы и производственное обеспечение",
    "Спец.одежда и СИЗ": "Материалы и производственное обеспечение",
    "Химреагенты": "Материалы и производственное обеспечение",
    "Прочие материалы": "Материалы и производственное обеспечение",
    "Прочие ГСМ": "Материалы и производственное обеспечение",
    "Командировочные расходы (независимо от основной деятельности)": "Командировочные и представительские расходы",
    "Курьерские услуги": "Транспорт и логистика",
    "Аренда офиса": "Аренда и содержание объектов",
    "Аренда автотранспорта": "Аренда и содержание объектов",
    "Консультационные услуги": "Консультационные и экспертные услуги",
    "Услуги по питанию персонала": "Расходы на персонал",
    "Аренда имущества": "Аренда и содержание объектов",
    "Интернет": "ИТ и связь",
    "Мобильная связь": "ИТ и связь",
    "Услуги связи": "ИТ и связь",
    "Комиссионное вознаграждение за реализацию продуктов ТАСКО": "Комиссионное вознаграждение",
    "Услуги автотранспорта (доставка нефтепродуктов)": "Транспорт и логистика",
    "Услуги по таможенному оформлению": "Таможенные и государственные услуги",
    "Налог на пользвание недрами и разбить на нефть, газ, конденсат": "Налоги, сборы и обязательные платежи",
    "Налог на имущество": "Налоги, сборы и обязательные платежи",
    "Налог на землю": "Налоги, сборы и обязательные платежи",
    "Налог на воду": "Налоги, сборы и обязательные платежи",
    "Налог на прибыль нерезидента по имп.усдуг. (не вычитаемая)": "Налоги, сборы и обязательные платежи",
    "Амортизация": "Амортизация и внеоборотные активы",
    "Канцтовары": "Материалы и производственное обеспечение",
    "Медикаменты": "Материалы и производственное обеспечение",
    "Продукты питания SANEG": "Материалы и производственное обеспечение",
    "Услуги спецтехники": "Ремонт и содержание имущества",
    "Прочие запасные части": "Материалы и производственное обеспечение",
    "Материалы для содержания территорий и помещений": "Ремонт и содержание имущества",
    "Производственный инструмент": "Материалы и производственное обеспечение",
    "Услуги автотранспорта прочие": "Транспорт и логистика",
    "Электрические расходные материалы": "Материалы и производственное обеспечение",
    "Резерв по отпускам": "Расходы на персонал",
    "Электроэнергия": "Коммунальные и энергетические расходы",
    "Прочие затраты на персонал": "Расходы на персонал",
    "ЕСП": "Налоги, сборы и обязательные платежи",
    "1.1.1. Материальные расходы": "Материалы и производственное обеспечение",
    "Зарабатная плата персонала": "Расходы на персонал",
    "Компенсации прочие": "Расходы на персонал",
    "Материальная помощь": "Расходы на персонал",
    "Премия к праздникам": "Расходы на персонал",
    "За использование природного газа": "Налоги, сборы и обязательные платежи",
    "2.2.6. Абонентская плата": "Абонентская плата(АЗС и АГНКС)",
    "Корректировка стоимости списания": "Материалы и производственное обеспечение",
    "Амортизация НМА": "Амортизация и внеоборотные активы",
    "Страхование прочие": "Штрафы, санкции, страхование и судебные расходы",
    "Списание ОС": "Списание ОС",
    "Судебные затраты": "Штрафы, санкции, страхование и судебные расходы",
    "Услуги по жд (не вычитаемая)": "Транспорт и логистика",
    "Расходы по отводу земель": "Прочие операционные и внереализационные расходы",
    "Прочие транспортные расходы": "Транспорт и логистика",
    "Представительские расходы": "Командировочные и представительские расходы",
    "Метрологическая сертификация и поверка измерительных приборов": "Консультационные и экспертные услуги",
    "Пеня по налогу": "Налоги, сборы и обязательные платежи",
    "Пенсионное отчисление по статье 12": "Налоги, сборы и обязательные платежи",
    "Акцизный налог": "Налоги, сборы и обязательные платежи",
    "Начислена амортизация": "Амортизация и внеоборотные активы",
    "Транспортировка сырья по трубопроводу": "Транспорт и логистика",
    "Потери товаров, продуктов при перевозке, хранении и реализации": "Транспорт и логистика",
    "Аренда земельного участка": "Аренда и содержание объектов",
    "Прочие командировочные расходы": "Командировочные и представительские расходы",
    "Электромонтажные работы": "Ремонт и содержание имущества",
    "Услуги по контролю входного к-во нефтепродуктов": "Консультационные и экспертные услуги",
    "Запасные части для автотранспорта": "Материалы и производственное обеспечение",
    "Комиссионное вознаграждение за реализацию продуктов SEGNUM": "Комиссионное вознаграждение",
    "Коммерческий блок": "Прочие операционные и внереализационные расходы",
    "Списание дебиторская и кредиторская задолженности": "Прочие операционные и внереализационные расходы",
    "Расходы по списанию дебиторской задолженности": "Прочие операционные и внереализационные расходы",
    "Обучение персонала": "Расходы на персонал",
    "Лабораторные исследования": "Консультационные и экспертные услуги",
    "Аудиторские услуги": "Консультационные и экспертные услуги",
    "Услуги полиграфии и типографии": "Прочие операционные и внереализационные расходы",
    "Медицинское обследование": "Расходы на персонал",
    "Пеня по налогам и сборам": "Налоги, сборы и обязательные платежи",
    "Гидроразрыв пласта": "Прочие расходы производственного характера",
    "Хранение и выдача нефтепродуктов": "Транспорт и логистика",
    "Прочие расходы производственного характера": "Прочие расходы производственного характера",
    "Продукты питания": "Материалы и производственное обеспечение",
    "Возврат товаров поставщику": "Прочие операционные и внереализационные расходы",
    "Взносы в профсоюз": "Расходы на персонал",
    "Страхование персонала": "Штрафы, санкции, страхование и судебные расходы",
    "Услуги по радиационной безопасности": "Консультационные и экспертные услуги",
    "Услуги аккредитации": "Консультационные и экспертные услуги",
    "Капитальный ремонт скважины": "Ремонт и содержание имущества",
    "Запасные части для оборудования": "Материалы и производственное обеспечение",
    "Услуги по пожарной и противофонтанной безопасности": "Консультационные и экспертные услуги",
    "Спонсорская помощь": "Спонсорская помощь",
    "Налог на прибыль нерезидента по имп.услуг.": "Налоги, сборы и обязательные платежи",
    "Перенос задолженности": "Прочие операционные и внереализационные расходы",
    "Подарки": "Прочие операционные и внереализационные расходы",
    "Таможенные услуги": "Таможенные и государственные услуги",
    "Комиссионное вознаграждение за реализацию продуктов Uzgastrade": "Комиссионное вознаграждение",
    "Ремонт оборудования": "Ремонт и содержание имущества",
    "Юридические услуги": "Консультационные и экспертные услуги",
    "Запчасти для оргтехники": "Материалы и производственное обеспечение",
    "Комиссионное вознаграждение банка": "Финансовые и банковские расходы",
    "Оценка имущества": "Консультационные и экспертные услуги",
    "Услуги консультанта": "Консультационные и экспертные услуги",
    "Специальный рентный налог на добычу полезных ископаемых": "Налоги, сборы и обязательные платежи",
    "Услуги автотранспорта и спец.техники": "Транспорт и логистика",
    "Компенсация потерь сельхоз производства": "Прочие операционные и внереализационные расходы",
    "Прочие ТМЦ и активы": "Материалы и производственное обеспечение",
    "Лицензирование": "Таможенные и государственные услуги",
    "Общие ТМЦ Карши": "Материалы и производственное обеспечение",
    "Переоценка балансовой стоимости": "Амортизация и внеоборотные активы",
    "Убытки от списания материальных ценностей": "Прочие операционные и внереализационные расходы",
    "Гидроразрывпласта": "Прочие расходы производственного характера",
    "Налог на прибыль нерезидента по имп.усдуг.": "Налоги, сборы и обязательные платежи",
    "Общие ТПП Карши": "Прочие операционные и внереализационные расходы",
    "Расходы по спесанию дебиторской задолженности": "Прочие операционные и внереализационные расходы",
    "Списания дебиторская и кредиторская задолженности": "Прочие операционные и внереализационные расходы"
}

## ***Меппинг статьи затрат Расходов периода***

In [34]:
overheads['Аналитика_свод'] = overheads['Аналитика'].map(expense_dict)


# ***Прочие доходы***

#№ ***Обработка исходника в Pandase***

In [63]:
df_other_income = def_load_and_combine(path_other_income, search_key = 'Период')
df_other_income = def_time_type(df_other_income, 'Период')
df_other_income = def_num_type(df_other_income, 'Unnamed: 5')
df_other_income = def_num_type(df_other_income, 'Unnamed: 8')
df_other_income = df_other_income.iloc[:, [0, 1, 2, 3,4,5,7,8]]
df_other_income = df_other_income.dropna(subset = 'Период')
df_other_income['Unnamed: 5'] = df_other_income['Unnamed: 5'].fillna(0)
df_other_income['Unnamed: 8'] = df_other_income['Unnamed: 8'].fillna(0)
df_other_income = df_other_income.rename(columns={"Период": "Дата","Аналитика Дт": "Аналитика_Дт",
                                                  "Аналитика Кт": "Аналитика_Кт", "Дебет": "Дебет_Счет", "Unnamed: 5": "Сумма_Дт", "Кредит": "Счет_Кт", "Unnamed: 8": "Сумма_Кт" })

## ***Обработка исходника в SQL***

In [62]:
query_other_income = """

      SELECT
            *
      FROM
            df_other_income

"""

df_other_income = duckdb.sql(query_other_income).df()
df_other_income

,Дата,Документ,Аналитика_Дт,Аналитика_Кт,Дебет_Счет,Сумма_Дт,Счет_Кт,Сумма_Кт
0,2025-01-01,Бухгалтерская справка Янги Узб252 от 01.01.202...,Абабакиров Абдупатто Абдувалиевич,Прочие доходы,6710.1,0.00,9391,0.19
1,2025-01-01,Бухгалтерская справка Янги Узб252 от 01.01.202...,Абашов Боходир Тошбаевич,Прочие доходы,6710.1,0.00,9391,1.19
2,2025-01-01,Бухгалтерская справка Янги Узб252 от 01.01.202...,Абдалимов Соли Самиевич,Прочие доходы,6710.1,0.00,9391,0.04
3,2025-01-01,Бухгалтерская справка Янги Узб252 от 01.01.202...,Абдиганиев Шохжахон Элбек угли,Прочие доходы,6710.1,0.00,9391,0.13
4,2025-01-01,Бухгалтерская справка Янги Узб252 от 01.01.202...,Абдиев Отабек Абдурашитович,Прочие доходы,6710.1,0.00,9391,0.88
...,...,...,...,...,...,...,...,...
3948,2025-12-31,Регламентная операция 00000000120 от 31.12.202...,Масло моторное М20А\r\nСклад готовой продукции...,Разницы стоимости возврата и фактической стоим...,2810.1,0.00,9390,"337,379,900.66"
3949,2025-12-31,Регламентная операция 00000000120 от 31.12.202...,Кокс нефтяной высокосернистый суммарный\r\nСкл...,Разницы стоимости возврата и фактической стоим...,2810.1,0.00,9390,"48,742,802.40"
3950,2025-12-31,Регламентная операция 00000000120 от 31.12.202...,Масло моторное М20А\r\nСклад готовой продукции...,Разницы стоимости возврата и фактической стоим...,2810.1,0.00,9390,"147,144,259.45"
3951,2025-12-31,Бухгалтерская справка 289 от 31.12.2025 23:59:59,Пенсионный фонд РУз\r\nСобирова Замира Маннабж...,Прочие доходы,4890.4,0.00,9390,"3,588,979.33"


# ***Экспорт и сохранение итоговых результатов***

In [51]:
exl_path = os.path.join(path_output, 'P&L_Saneg.xlsx')

with pd.ExcelWriter(exl_path) as writer:
    # reven_cost_of_sale.to_excel(writer, sheet_name='Выр_себес', index=False)
    # cost_of_sale.to_excel(writer, sheet_name='Себестоимость', index=False)
    # revenue.to_excel(writer, sheet_name='Выручка', index=False)
    # overheads.to_excel(writer, sheet_name='Расходы_периода', index=False)
    df_other_income.to_excel(writer, sheet_name='Прочие_доходы', index=False)